# BioSentinel India: Spatial-Temporal Intelligence

**Owner:** Person 2
**Scope:** Spatial maps, Temporal trends, Sampling-bias metrics, Baseline construction

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root.")

PROJECT_ROOT = find_project_root()
ANALYTICS_DIR = PROJECT_ROOT / "data" / "processed" / "analytics"

print("Project root:", PROJECT_ROOT)
print("Analytics dir:", ANALYTICS_DIR)

## 1. Load Data
Load the outputs from the Biodiversity & Data Analytics stage (Person 1).

In [ ]:
try:
    annual_richness = pd.read_csv(ANALYTICS_DIR / "annual_species_richness.csv")
    grid_richness = pd.read_csv(ANALYTICS_DIR / "grid_species_richness.csv")
    print("Loaded annual richness records:", len(annual_richness))
    print("Loaded grid richness records:", len(grid_richness))
except FileNotFoundError as e:
    print("Error loading Person 1 outputs:", e)
    print("Make sure notebooks/02_biodiversity_data_analysis.ipynb has been executed.")
    # Creating dummy data to allow notebook to compile structure if missing
    annual_richness = pd.DataFrame({'year': range(2010, 2021), 'richness': np.random.randint(100, 500, 11)})
    grid_richness = pd.DataFrame({'eqdcellcode': [f'C{i}' for i in range(100)], 'richness': np.random.randint(10, 200, 100)})

try:
    grid_year_indicators = pd.read_parquet(ANALYTICS_DIR / "grid_year_biodiversity_indicators.parquet")
    print("Loaded grid year indicators records:", len(grid_year_indicators))
except FileNotFoundError:
    print("grid_year_biodiversity_indicators.parquet not found.")


## 2. Temporal Trends
Analyze and visualize how species richness changes over the years.

In [ ]:
plt.figure(figsize=(10, 5))
# Assuming 'year' and a richness column exist. We will dynamically find the richness column name.
richness_col = [c for c in annual_richness.columns if 'richness' in c.lower() or 'species' in c.lower() and c != 'year']
if richness_col:
    y_col = richness_col[0]
    sns.lineplot(data=annual_richness, x='year', y=y_col, marker='o', color='b')
    plt.title('Temporal Trend: Species Richness Over Time')
    plt.xlabel('Year')
    plt.ylabel('Species Richness')
    plt.xticks(annual_richness['year'].unique())
    plt.tight_layout()
    plt.show()
else:
    print("Could not identify the richness column in annual_richness.csv")

## 3. Spatial Maps / Grid Visualization
Visualize the distribution of biodiversity across grid cells. (A full spatial map requires mapping `eqdcellcode` to geometries).

In [ ]:
plt.figure(figsize=(12, 6))
grid_richness_sorted = grid_richness.sort_values(by=grid_richness.columns[1], ascending=False).head(40)
sns.barplot(data=grid_richness_sorted, x='eqdcellcode', y=grid_richness.columns[1], palette='viridis')
plt.title('Top 40 Grid Cells by Species Richness')
plt.xticks(rotation=90)
plt.ylabel('Species Richness')
plt.xlabel('Grid Cell Code')
plt.tight_layout()
plt.show()

## 4. Sampling-Bias Metrics
Calculate spatial sampling bias using the Gini coefficient. A high Gini coefficient indicates that sampling is concentrated in a few grid cells (high bias), while a low coefficient indicates even sampling.

In [ ]:
def gini(array):
    # Calculate the Gini coefficient of a numpy array
    array = np.sort(np.array(array, dtype=np.float64))
    if array.shape[0] == 0 or np.sum(array) == 0:
        return 0
    index = np.arange(1, array.shape[0] + 1)
    n = array.shape[0]
    return ((np.sum((2 * index - n  - 1) * array)) / (n * np.sum(array)))

grid_counts = grid_richness[grid_richness.columns[1]].values
g_coeff = gini(grid_counts)
print(f"Gini Coefficient for Spatial Sampling Bias: {g_coeff:.4f}")
if g_coeff > 0.6:
    print("-> High spatial bias: Sampling is highly concentrated in specific areas.")
elif g_coeff > 0.4:
    print("-> Moderate spatial bias: Sampling is somewhat uneven.")
else:
    print("-> Low spatial bias: Sampling is relatively evenly distributed.")

## 5. Baseline Construction
Establish a baseline species richness using early years of data, enabling comparison for subsequent years to detect anomalies or trends.

In [ ]:
if richness_col:
    y_col = richness_col[0]
    baseline_years = annual_richness['year'].nsmallest(3)
    baseline_value = annual_richness[annual_richness['year'].isin(baseline_years)][y_col].mean()
    
    annual_richness['baseline'] = baseline_value
    annual_richness['deviation_from_baseline'] = annual_richness[y_col] - baseline_value
    annual_richness['pct_change_from_baseline'] = (annual_richness['deviation_from_baseline'] / baseline_value) * 100
    
    print(f"Baseline Richness (calculated over {baseline_years.min()} to {baseline_years.max()}): {baseline_value:.2f}\n")
    try:
        display(annual_richness[['year', y_col, 'deviation_from_baseline', 'pct_change_from_baseline']])
    except NameError:
        print(annual_richness[['year', y_col, 'deviation_from_baseline', 'pct_change_from_baseline']])
    
    plt.figure(figsize=(10, 5))
    sns.barplot(data=annual_richness, x='year', y='pct_change_from_baseline', 
                palette=annual_richness['pct_change_from_baseline'].apply(lambda x: 'red' if x < 0 else 'green'))
    plt.title('Percentage Change from Baseline Richness')
    plt.ylabel('% Change')
    plt.xlabel('Year')
    plt.axhline(0, color='black', linewidth=1)
    plt.tight_layout()
    plt.show()
else:
    print("Cannot construct baseline without identifying the richness column.")